### Install libraries (transformers 4.52.4 is required because v5 removed the summarization pipeline).

In [1]:
!pip install -q transformers==4.52.4 youtube-transcript-api fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 11.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.6 MB/s eta 0:00:00:00:01


### Extract the video ID from the URL and fetch the video's transcript.

In [2]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi


def extract_video_id(url: str) -> str:
    """
    Extract the YouTube video ID from a URL.
    Raises ValueError if no 'v' parameter is found.
    """
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    if host.startswith("www."):
        host = host[4:]

    if host == "youtu.be":
        video_id = parsed.path.lstrip("/").split("/")[0]
        if video_id:
            return video_id

    if host in ("youtube.com", "m.youtube.com", "music.youtube.com"):
        qs = parse_qs(parsed.query)
        if qs.get("v"):
            return qs["v"][0]

        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in ("shorts", "embed", "live", "v"):
            return parts[1]

    raise ValueError(f"No video id found in URL: {url}")


url = "https://www.youtube.com/watch?v=o-wv_szZ0V0" # <-------------------------------- 'Put The Video Link Here'
video_id = extract_video_id(url)
api = YouTubeTranscriptApi()
fetched = api.fetch(video_id, languages=["en"])
text = "\n".join(snippet.text for snippet in fetched)


### Clean the transcript by removing noise like `[music]` and `>>` and joining broken lines.

In [3]:
import re

def clean_transcript(snippets):
    text = " ".join(s.text.replace("\n", " ") for s in snippets)
    text = re.sub(r"\[.*?\]", " ", text)     
    text = text.replace(">>", "\n")         
    text = re.sub(r"[ \t]+", " ", text)     
    text = re.sub(r"\n\s*", "\n", text).strip()
    return text

clean_text = clean_transcript(fetched)
print("قبل:", len(text.split()), "كلمة | بعد:", len(clean_text.split()), "كلمة")
print(clean_text[:1500])

قبل: 7388 كلمة | بعد: 7289 كلمة
There's been a lot of misinformation about AI. 
This is Andrew. He co-founded Google Brain and Coursera. His machine learning course has reached millions of learners and he is one of the most influential voices in AI today. 
So, a handful of leading AI companies have been very loud voices fearmongering around AI to try to get regulations passed. This Trumpian fear-based messaging has skewed societal's perception to be really negative on AI. 
People talk to me about data centers and job loss. 
Maybe AI could do 30-40% of many jobs. And what that means is well, that 60% that a human does has become even more valuable. 
What about loss of human control over AI? 
I think about something else that we can't control. 
I think you're one of the voices in AI who comes with a huge background in machine learning and teaching AI. And also you're a positive voice cuz this is something that I've been seeing especially this summer, how polarized the society has become 

### Load a BART model fine-tuned on dialogues, which doesn't invent names like the original bart-large-cnn.

In [13]:
from transformers import pipeline

summarizer = pipeline("summarization", model="philschmid/bart-large-cnn-samsum", device_map="auto")
GEN_ARGS = dict(do_sample=False, num_beams=2, no_repeat_ngram_size=3, truncation=True)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (8904 > 1024). Running this sequence through the model will result in indexing errors


12 chunks
INFO:     156.199.196.62:0 - "POST /summarize HTTP/1.1" 200 OK
2 chunks
INFO:     156.199.196.62:0 - "POST /summarize HTTP/1.1" 200 OK


### Split the text into overlapping chunks by tokens, since the model only accepts 1024 tokens.

In [5]:
def chunk_by_tokens(text, tokenizer, max_tokens=900, overlap=100):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    chunks, step = [], max_tokens - overlap
    for start in range(0, len(ids), step):
        chunks.append(tokenizer.decode(ids[start:start + max_tokens]))
        if start + max_tokens >= len(ids):
            break
    return chunks

### Summarize one text with a summary length based on the input length.

In [6]:
def summarize_one(text, cap):
    n = len(summarizer.tokenizer(text)["input_ids"])
    max_len = max(30, min(cap, int(n * 0.5)))  
    min_len = max(15, int(max_len * 0.4))
    return summarizer(text, max_length=max_len, min_length=min_len, **GEN_ARGS)[0]["summary_text"]

### Summarize each chunk (map), then combine them into one final summary (reduce).

In [7]:
def summarize_long_text(text, summarizer, max_tokens=900):
    tok = summarizer.tokenizer
    chunks = chunk_by_tokens(text, tok, max_tokens)
    print(f"{len(chunks)} chunks")

    partials = [summarize_one(c, cap=120) for c in chunks]          # Map
    combined = " ".join(partials)

    if len(tok(combined)["input_ids"]) > max_tokens:
        return summarize_long_text(combined, summarizer, max_tokens)

    final = summarize_one(combined, cap=250)                         # Reduce
    return final, partials

### Run the summarizer and print the final summary and each part's summary.

In [8]:
#final, partials = summarize_long_text(clean_text, summarizer)

#print("===== Final Summary =====\n", final)
#print("\n===== Partial Summary =====")
#for i, p in enumerate(partials, 1):
   # print(f"{i}. {p}")

### Wrap the whole pipeline in one function that takes a URL, to be used later by the API.

In [9]:
def summarize_video(url):
    video_id = extract_video_id(url)
    fetched = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    clean_text = clean_transcript(fetched)
    final, partials = summarize_long_text(clean_text, summarizer)
    return {"video_id": video_id, "summary": final, "sections": partials}


#result = summarize_video("https://www.youtube.com/watch?v=o-wv_szZ0V0")
#print(result["summary"])

In [10]:
NGROK_TOKEN = "ADD_NGROK_TOKEN"   
API_KEY = "secret123"             

In [11]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/summarize")
async def summarize(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    return summarize_video(data.get("url", ""))

In [12]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://rehab-unpopular-recast.ngrok-free.dev                                      


INFO:     Started server process [83]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:41789 (Press CTRL+C to quit)
